# Análisis de Energía Solar - Popayán (2023-2026)
Este notebook realiza un análisis exploratorio completo (EDA) de los datos del sistema solar de Popayán, entrena una Red Neuronal para la predicción de generación de energía solar en noviembre y diciembre de 2026, y expone los resultados a través de un Dashboard interactivo en Gradio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Cargar el dataset
file_path = '/content/dataset_sistema_solar_popayan_2023_2026_v6.csv'
df = pd.read_csv(file_path)

# Mostrar información básica
print("Dimensiones del dataset:", df.shape)
print("\nColumnas del dataset:")
print(df.info())
print("\nPrimeras filas:")
display(df.head())

Dimensiones del dataset: (29256, 16)

Columnas del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29256 entries, 0 to 29255
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Timestamp           29256 non-null  object 
 1   P_DC_PV_W           29256 non-null  float64
 2   V_Batt_V            29256 non-null  float64
 3   I_Batt_A            29256 non-null  float64
 4   SoC_Batt_Pct        29256 non-null  float64
 5   V_AC_L1_V           29256 non-null  float64
 6   V_AC_L2_V           29256 non-null  float64
 7   V_AC_L3_V           29256 non-null  float64
 8   I_AC_L1_A           29256 non-null  float64
 9   I_AC_L2_A           29256 non-null  float64
 10  I_AC_L3_A           29256 non-null  float64
 11  Grid_Status         29256 non-null  object 
 12  Temp_Inverter_C     29256 non-null  float64
 13  Status_Code         29256 non-null  int64  
 14  Alarm_Event         29256 non-null  object

,Timestamp,P_DC_PV_W,V_Batt_V,I_Batt_A,SoC_Batt_Pct,V_AC_L1_V,V_AC_L2_V,V_AC_L3_V,I_AC_L1_A,I_AC_L2_A,I_AC_L3_A,Grid_Status,Temp_Inverter_C,Status_Code,Alarm_Event,Fault_Duration_Min
0,2023-07-01 00:00:00,0.0,49.99,-32.0,66.6,119.4,119.1,119.0,12.28,11.94,12.07,On-Grid (Red/Descarga Noche),25.1,100,NONE,0
1,2023-07-01 01:00:00,0.0,49.33,-22.6,61.1,121.1,120.1,119.2,11.94,12.17,12.22,On-Grid (Red/Descarga Noche),25.1,100,NONE,0
2,2023-07-01 02:00:00,0.0,48.70,-13.9,55.8,118.8,118.4,121.2,11.52,11.60,11.72,On-Grid (Red/Descarga Noche),25.1,100,NONE,0
3,2023-07-01 03:00:00,0.0,47.94,-26.8,49.5,121.3,121.7,121.8,12.30,12.11,12.26,On-Grid (Red/Descarga Noche),25.1,100,NONE,0
4,2023-07-01 04:00:00,0.0,47.28,-25.4,44.0,119.8,119.8,118.2,11.97,12.11,11.99,On-Grid (Red/Descarga Noche),25.1,100,NONE,0


## 1. Análisis Exploratorio de Datos (EDA) y Preparación de Datos
Procederemos a examinar la presencia de nulos, estadísticas descriptivas, formatear la variable de fecha y realizar el análisis gráfico.

In [ ]:
# Conversión de fecha y extracción de características temporales
fecha_col = [col for col in df.columns if 'timestamp' in col.lower() or 'fecha' in col.lower() or 'date' in col.lower()][0]
df[fecha_col] = pd.to_datetime(df[fecha_col])

df['Anio'] = df[fecha_col].dt.year
df['Mes'] = df[fecha_col].dt.month
df['Dia'] = df[fecha_col].dt.day
df['Hora'] = df[fecha_col].dt.hour

# Identificar columna objetivo (P_DC_PV_W es la generación de potencia DC de los paneles fotovoltaicos)
target_col = 'P_DC_PV_W'
print(f"Columna objetivo identificada: {target_col}")

# Estadísticas descriptivas
print("\nEstadísticas descriptivas de variables numéricas:")
display(df.describe())

Columna objetivo identificada: P_DC_PV_W

Estadísticas descriptivas de variables numéricas:


,Timestamp,P_DC_PV_W,V_Batt_V,I_Batt_A,SoC_Batt_Pct,V_AC_L1_V,V_AC_L2_V,V_AC_L3_V,I_AC_L1_A,I_AC_L2_A,I_AC_L3_A,Temp_Inverter_C,Status_Code,Fault_Duration_Min,Anio,Mes,Dia,Hora
count,29256,29256.00000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000,29256.000000
mean,2025-03-01 11:30:00,2756.08911,50.502167,6.111622,70.859212,119.991715,120.001812,120.005052,12.271846,12.271729,12.272795,33.990375,100.328548,0.170221,2024.647252,6.721083,15.747334,11.500000
min,2023-07-01 00:00:00,0.00000,42.000000,-48.600000,0.000000,118.200000,118.200000,118.200000,0.000000,0.000000,0.000000,25.100000,100.000000,0.000000,2023.000000,1.000000,1.000000,0.000000
25%,2024-04-30 17:45:00,0.00000,49.100000,-24.300000,59.200000,119.100000,119.100000,119.100000,11.970000,11.970000,11.970000,25.100000,100.000000,0.000000,2024.000000,4.000000,8.000000,5.750000
50%,2025-03-01 11:30:00,0.00000,50.660000,-13.700000,72.200000,120.000000,120.000000,120.000000,12.280000,12.280000,12.280000,25.100000,100.000000,0.000000,2025.000000,7.000000,16.000000,11.500000
75%,2025-12-31 05:15:00,5919.47500,52.340000,40.300000,86.200000,120.900000,120.900000,120.900000,12.590000,12.590000,12.590000,44.300000,100.000000,0.000000,2025.000000,10.000000,23.000000,17.250000
max,2026-10-31 23:00:00,13058.60000,54.000000,89.400000,100.000000,121.800000,121.800000,121.800000,13.150000,13.150000,13.150000,58.700000,501.000000,360.000000,2026.000000,12.000000,31.000000,23.000000
std,NaN,3592.80194,2.207977,35.217287,18.387573,1.039025,1.043943,1.037697,0.506153,0.507058,0.507061,10.460359,11.466483,7.383464,1.014400,3.343064,8.809320,6.922305


In [ ]:
# Generación de Gráficos para el EDA y almacenamiento como imágenes para usarlas en Gradio
os.makedirs('/content/eda_plots', exist_ok=True)

# 1. Distribución Univariada del Target
plt.figure(figsize=(8, 5))
sns.histplot(df[target_col], kde=True, color='orange')
plt.title(f'Distribución de la Variable Objetivo: {target_col}')
plt.xlabel(target_col)
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('/content/eda_plots/dist_univariada.png')
plt.close()

# 2. Análisis Bivariado: Generación promedio por Mes
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x='Mes', y=target_col, errorbar=None, palette='viridis', hue='Mes', legend=False)
plt.title('Generación Promedio de Energía por Mes del Año')
plt.xlabel('Mes')
plt.ylabel('Generación Promedio')
plt.tight_layout()
plt.savefig('/content/eda_plots/bivariado_mes.png')
plt.close()

# 3. Matriz de Correlación
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación del Dataset')
plt.tight_layout()
plt.savefig('/content/eda_plots/matriz_correlacion.png')
plt.close()

print("Gráficos del EDA guardados con éxito en '/content/eda_plots'")

Gráficos del EDA guardados con éxito en '/content/eda_plots'


## 2. Entrenamiento de la Red Neuronal (ANN)
Entrenaremos un regresor usando una Red Neuronal para predecir la generación de energía solar.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Definir variables predictoras y objetivo
numeric_cols = df.select_dtypes(include=[np.number]).columns
features = [col for col in numeric_cols if col != target_col]
X = df[features]
y = df[target_col]

# División de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalado de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Definición del modelo de Red Neuronal
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.1),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1) # Salida lineal para regresión
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Entrenamiento del modelo
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=64,
    verbose=0
)

print("¡Modelo entrenado exitosamente!")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


¡Modelo entrenado exitosamente!


## 3. Métricas de Evaluación del Modelo e Histograma de Errores

In [ ]:
# Predicciones en Test
y_pred = model.predict(X_test_scaled).flatten()

# Cálculo de Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Métricas de Regresión:")
print(f"MSE (Error Cuadrático Medio): {mse:.4f}")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse:.4f}")
print(f"MAE (Error Absoluto Medio): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")

# Histograma de Errores
errores = y_test - y_pred
plt.figure(figsize=(8, 5))
sns.histplot(errores, kde=True, color='crimson')
plt.axvline(x=0, color='black', linestyle='--')
plt.title('Histograma de Errores de Predicción')
plt.xlabel('Error (Real - Predicción)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.savefig('/content/eda_plots/histograma_errores.png')
plt.close()

183/183 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Métricas de Regresión:
MSE (Error Cuadrático Medio): 1778.4141
RMSE (Raíz del Error Cuadrático Medio): 42.1712
MAE (Error Absoluto Medio): 20.8926
R2 Score: 0.9999


## 4. Ejemplo de Predicción de Generación para Noviembre y Diciembre de 2026
Crearemos escenarios predictivos basados en los promedios del histórico del dataset para dichos meses en el año 2026.

In [ ]:
# Generación de escenario sintético basado en promedios históricos para Noviembre y Diciembre de 2026
promedio_features = df[features].mean().to_dict()

# Creamos una fila típica para noviembre de 2026 y otra para diciembre de 2026
escenario_nov = promedio_features.copy()
escenario_nov['Anio'] = 2026
escenario_nov['Mes'] = 11

escenario_dic = promedio_features.copy()
escenario_dic['Anio'] = 2026
escenario_dic['Mes'] = 12

df_predicciones = pd.DataFrame([escenario_nov, escenario_dic])
X_pred_scaled = scaler.transform(df_predicciones)

predicciones_2026 = model.predict(X_pred_scaled).flatten()

print(f"Predicción de Generación Promedio para Noviembre 2026: {predicciones_2026[0]:.2f} W")
print(f"Predicción de Generación Promedio para Diciembre 2026: {predicciones_2026[1]:.2f} W")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Predicción de Generación Promedio para Noviembre 2026: 762.76 W
Predicción de Generación Promedio para Diciembre 2026: 775.45 W


## 5. Implementación del Dashboard Interactivo con Gradio

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

# Preparación de explicaciones y conclusiones
explicacion_proyecto = """
### Explicación del Ejercicio
Este proyecto implementa una metodología integral para el análisis y la estimación de energía solar fotovoltaica en Popayán para el periodo final de 2026.
1. **EDA**: Identificación de patrones de radiación y generación a lo largo de las distintas épocas del año.
2. **Redes Neuronales**: Se utiliza un regresor de Deep Learning con capas densas optimizadas con Dropout para evitar el sobreajuste, logrando una excelente aproximación con los patrones estacionales locales.
"""

conclusiones_ejercicio = f"""
### Conclusiones del Ejercicio
1. **Estacionalidad Clara**: El dataset evidencia de manera robusta cómo la época del año influye de manera drástica en la generación fotovoltaica.
2. **Rendimiento de la Red Neuronal**: Con un Score R2 obtenido de **{r2:.4f}**, el modelo de Redes Neuronales es altamente confiable para predecir la producción de los sistemas fotovoltaicos en Popayán.
3. **Proyecciones para fin de año 2026**: Las proyecciones estiman una generación promedio para Noviembre de **{predicciones_2026[0]:.2f} W** y para Diciembre de **{predicciones_2026[1]:.2f} W**.
"""

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ☀️ Dashboard Inteligente: Sistema Solar Popayán (2023-2026)")

    with gr.Tab("Explicación del Proyecto"):
        gr.Markdown(explicacion_proyecto)

    with gr.Tab("Vista del Dataframe"):
        gr.Markdown("### Muestra de las primeras 10 filas de datos originales")
        gr.DataFrame(df.head(10))

    with gr.Tab("Estadísticas Descriptivas"):
        gr.Markdown("### Resumen estadístico de las variables principales")
        gr.DataFrame(df.describe())

    with gr.Tab("Análisis Univariado y Bivariado"):
        gr.Markdown("### Comportamiento de la Generación")
        with gr.Row():
            gr.Image("/content/eda_plots/dist_univariada.png", label="Distribución Univariada")
            gr.Image("/content/eda_plots/bivariado_mes.png", label="Generación por Mes (Bivariado)")

    with gr.Tab("Correlación y Dependencias"):
        gr.Markdown("### Matriz de correlación general")
        gr.Image("/content/eda_plots/matriz_correlacion.png")

    with gr.Tab("Métricas & Modelo ANN"):
        gr.Markdown("### Evaluación del Desempeño del Modelo")
        with gr.Row():
            gr.Markdown(f"""
            - **MSE (Error Cuadrático Medio):** {mse:.4f}
            - **RMSE:** {rmse:.4f}
            - **MAE (Error Absoluto Medio):** {mae:.4f}
            - **R² Score (Coeficiente de Determinación):** {r2:.4f}
            """)
            gr.Image("/content/eda_plots/histograma_errores.png", label="Histograma de Errores")

    with gr.Tab("Predicciones Nov-Dic 2026 & Conclusiones"):
        gr.Markdown("### Estimaciones de Generación y Conclusiones del Ejercicio")
        with gr.Row():
            gr.Markdown(f"""
            ### Predicciones para Fin de Año 2026
            - 📅 **Noviembre 2026:** {predicciones_2026[0]:.2f} W
            - 📅 **Diciembre 2026:** {predicciones_2026[1]:.2f} W
            """)
        gr.Markdown(conclusiones_ejercicio)

# Lanzar la app
demo.launch(debug=True, share=True)

/tmp/ipykernel_1773/3341743468.py:18: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fb1915495916aba28b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
